<a href="https://colab.research.google.com/github/sohamajwani/Traffic-Sign-CNN/blob/main/traditional_ml_models_ccf_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, matthews_corrcoef,
    roc_auc_score, average_precision_score
)

In [2]:
df = pd.read_csv('creditcard.csv')
df.dropna(inplace=True)
df.drop('Time', axis=1, inplace=True, errors='ignore')

###Logistic Regression

In [5]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=101, stratify=y
)

In [6]:
scaler = RobustScaler()
X_train['Amount'] = scaler.fit_transform(X_train['Amount'].values.reshape(-1, 1))
X_test['Amount'] = scaler.transform(X_test['Amount'].values.reshape(-1, 1))

In [7]:
#Hyperparameter tuning using GridSearchCV optimized for PR-AUC

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2']
}

# Using StratifiedKFold for the grid search CV split
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

grid_search = GridSearchCV(
    estimator=LogisticRegression(solver='liblinear', class_weight='balanced', random_state=101),
    param_grid=param_grid,
    scoring='average_precision',  # Focuses tuning on maximizing PR-AUC
    cv=cv_strategy,
    n_jobs=-1
)

In [8]:
print("Tuning Logistic Regression model parameters...")
grid_search.fit(X_train, y_train)
best_log_model = grid_search.best_estimator_
print(f"Best Hyperparameters Found: {grid_search.best_params_}")

Tuning Logistic Regression model parameters...
Best Hyperparameters Found: {'C': 0.01, 'penalty': 'l1'}


In [9]:
# predictions
predictions = best_log_model.predict(X_test)
y_scores = best_log_model.predict_proba(X_test)[:, 1]

In [10]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()

In [11]:
metrics_log_reg = {
    'Accuracy': accuracy_score(y_test, predictions),
    'Precision (Class 1)': precision_score(y_test, predictions, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions),
    'ROC-AUC': roc_auc_score(y_test, y_scores),
    'PR-AUC': average_precision_score(y_test, y_scores, pos_label=1)
}

In [12]:
log_regression_results = pd.DataFrame(pd.Series(metrics_log_reg), columns=['Best Logistic Regression'])
print("\n--- Evaluation Parameters ---")
print(log_regression_results.round(4))


--- Evaluation Parameters ---
                     Best Logistic Regression
Accuracy                               0.9821
Precision (Class 1)                    0.1293
Recall (Class 1)                       0.9677
F1-Score (Class 1)                     0.2281
TNR                                    0.9821
MCC                                    0.3503
ROC-AUC                                0.9944
PR-AUC                                 0.7549


###Decision Tree

In [13]:
from sklearn.tree import DecisionTreeClassifier

In [14]:
param_grid_dt = {
    'max_depth': [5, 8, 12, None],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy']
}

In [15]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=101)

In [16]:
grid_search_dt = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=101),
    param_grid=param_grid_dt,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

print("Tuning Decision Tree parameters on your existing train set...")
grid_search_dt.fit(X_train, y_train)
best_dt_model = grid_search_dt.best_estimator_
print(f"Best Hyperparameters Found: {grid_search_dt.best_params_}")

Tuning Decision Tree parameters on your existing train set...
Best Hyperparameters Found: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 8, 'min_samples_split': 5}


In [17]:
predictions_dt = best_dt_model.predict(X_test)
y_scores_dt = best_dt_model.predict_proba(X_test)[:, 1]

In [18]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions_dt).ravel()

In [19]:
metrics_dt = {
    'Accuracy': accuracy_score(y_test, predictions_dt),
    'Precision (Class 1)': precision_score(y_test, predictions_dt, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions_dt, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions_dt, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions_dt),
    'ROC-AUC': roc_auc_score(y_test, y_scores_dt),
    'PR-AUC': average_precision_score(y_test, y_scores_dt, pos_label=1)
}

In [20]:
dt_results = pd.DataFrame(pd.Series(metrics_dt), columns=['Best Decision Tree'])
print("\n--- Decision Tree Evaluation Parameters ---")
print(dt_results.round(4))


--- Decision Tree Evaluation Parameters ---
                     Best Decision Tree
Accuracy                         0.9988
Precision (Class 1)              0.7931
Recall (Class 1)                 0.7419
F1-Score (Class 1)               0.7667
TNR                              0.9995
MCC                              0.7665
ROC-AUC                          0.8709
PR-AUC                           0.7426


###KNN

In [21]:
from sklearn.neighbors import KNeighborsClassifier

In [22]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7],
    'weights': ['uniform', 'distance'],
    'metric': ['minkowski', 'euclidean']
}

In [23]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=101)

In [24]:
grid_search_knn = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid_knn,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

print("Tuning KNN parameters... (This may take a few minutes due to distance calculations)")
grid_search_knn.fit(X_train, y_train)
best_knn_model = grid_search_knn.best_estimator_
print(f"Best Hyperparameters Found: {grid_search_knn.best_params_}")

Tuning KNN parameters... (This may take a few minutes due to distance calculations)
Best Hyperparameters Found: {'metric': 'minkowski', 'n_neighbors': 7, 'weights': 'distance'}


In [25]:
predictions_knn = best_knn_model.predict(X_test)
y_scores_knn = best_knn_model.predict_proba(X_test)[:, 1]

In [26]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions_knn).ravel()

In [27]:
metrics_knn = {
    'Accuracy': accuracy_score(y_test, predictions_knn),
    'Precision (Class 1)': precision_score(y_test, predictions_knn, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions_knn, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions_knn, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions_knn),
    'ROC-AUC': roc_auc_score(y_test, y_scores_knn),
    'PR-AUC': average_precision_score(y_test, y_scores_knn, pos_label=1)
}

In [28]:
knn_results = pd.DataFrame(pd.Series(metrics_knn), columns=['Best KNN'])
print("\n--- KNN Evaluation Parameters ---")
print(knn_results.round(4))


--- KNN Evaluation Parameters ---
                     Best KNN
Accuracy               0.9991
Precision (Class 1)    0.9565
Recall (Class 1)       0.7097
F1-Score (Class 1)     0.8148
TNR                    0.9999
MCC                    0.8235
ROC-AUC                0.9676
PR-AUC                 0.9024


###SVM

In [29]:
from sklearn.svm import SVC

In [30]:
param_grid_svm = {
    'C': [0.1, 1, 10],
    'class_weight': ['balanced', None]
}

In [31]:
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=101)

In [32]:
grid_search_svm = GridSearchCV(
    estimator=SVC(kernel='linear', probability=True, random_state=101),
    param_grid=param_grid_svm,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

print("Tuning SVM parameters... (This may take a brief moment as it calculates margins)")
grid_search_svm.fit(X_train, y_train)
best_svm_model = grid_search_svm.best_estimator_
print(f"Best Hyperparameters Found: {grid_search_svm.best_params_}")

Tuning SVM parameters... (This may take a brief moment as it calculates margins)
Best Hyperparameters Found: {'C': 0.1, 'class_weight': None}


In [33]:
predictions_svm = best_svm_model.predict(X_test)
y_scores_svm = best_svm_model.predict_proba(X_test)[:, 1]

In [34]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions_svm).ravel()

In [35]:
metrics_svm = {
    'Accuracy': accuracy_score(y_test, predictions_svm),
    'Precision (Class 1)': precision_score(y_test, predictions_svm, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions_svm, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions_svm, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions_svm),
    'ROC-AUC': roc_auc_score(y_test, y_scores_svm),
    'PR-AUC': average_precision_score(y_test, y_scores_svm, pos_label=1)
}

In [36]:
svm_results = pd.DataFrame(pd.Series(metrics_svm), columns=['Best SVM'])
print("\n--- SVM Evaluation Parameters ---")
print(svm_results.round(4))


--- SVM Evaluation Parameters ---
                     Best SVM
Accuracy               0.9987
Precision (Class 1)    0.8077
Recall (Class 1)       0.6774
F1-Score (Class 1)     0.7368
TNR                    0.9996
MCC                    0.7390
ROC-AUC                0.9451
PR-AUC                 0.7133


###Random Forest

In [37]:
from sklearn.ensemble import RandomForestClassifier

In [38]:
param_grid_rf = {
    'n_estimators': [50, 100],
    'max_depth': [8, 12, None],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample']
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=101)

In [39]:
grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=101, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

print("Tuning Random Forest... (Growing trees in parallel, hang tight!)")
grid_search_rf.fit(X_train, y_train)
best_rf_model = grid_search_rf.best_estimator_
print(f"Best Hyperparameters Found: {grid_search_rf.best_params_}")

Tuning Random Forest... (Growing trees in parallel, hang tight!)
Best Hyperparameters Found: {'class_weight': 'balanced_subsample', 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 100}


In [40]:
predictions_rf = best_rf_model.predict(X_test)
y_scores_rf = best_rf_model.predict_proba(X_test)[:, 1]

In [41]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions_rf).ravel()

metrics_rf = {
    'Accuracy': accuracy_score(y_test, predictions_rf),
    'Precision (Class 1)': precision_score(y_test, predictions_rf, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions_rf, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions_rf, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions_rf),
    'ROC-AUC': roc_auc_score(y_test, y_scores_rf),
    'PR-AUC': average_precision_score(y_test, y_scores_rf, pos_label=1)
}

In [42]:
rf_results = pd.DataFrame(pd.Series(metrics_rf), columns=['Best Random Forest'])
print("\n--- Random Forest Evaluation Parameters ---")
print(rf_results.round(4))


--- Random Forest Evaluation Parameters ---
                     Best Random Forest
Accuracy                         0.9990
Precision (Class 1)              0.9545
Recall (Class 1)                 0.6774
F1-Score (Class 1)               0.7925
TNR                              0.9999
MCC                              0.8037
ROC-AUC                          0.9835
PR-AUC                           0.9388


###XGBoost

In [44]:
import xgboost as xgb

In [45]:
scale_pos_weight_value = (y_train == 0).sum() / (y_train == 1).sum()

In [46]:
param_grid_xgb = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 150],
    'scale_pos_weight': [1, scale_pos_weight_value] # Test both default and heavily weighted structures
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=101)

In [47]:
grid_search_xgb = GridSearchCV(
    estimator=xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=101),
    param_grid=param_grid_xgb,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

print("Tuning XGBoost... (Optimizing error corrections sequentially)")
grid_search_xgb.fit(X_train, y_train)
best_xgb_model = grid_search_xgb.best_estimator_
print(f"Best Hyperparameters Found: {grid_search_xgb.best_params_}")

Tuning XGBoost... (Optimizing error corrections sequentially)
Best Hyperparameters Found: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 150, 'scale_pos_weight': 1}


In [48]:
predictions_xgb = best_xgb_model.predict(X_test)
y_scores_xgb = best_xgb_model.predict_proba(X_test)[:, 1]

In [49]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions_xgb).ravel()

metrics_xgb = {
    'Accuracy': accuracy_score(y_test, predictions_xgb),
    'Precision (Class 1)': precision_score(y_test, predictions_xgb, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions_xgb, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions_xgb, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions_xgb),
    'ROC-AUC': roc_auc_score(y_test, y_scores_xgb),
    'PR-AUC': average_precision_score(y_test, y_scores_xgb, pos_label=1)
}

In [50]:
xgb_results = pd.DataFrame(pd.Series(metrics_xgb), columns=['Best XGBoost'])
print("\n--- XGBoost Evaluation Parameters ---")
print(xgb_results.round(4))


--- XGBoost Evaluation Parameters ---
                     Best XGBoost
Accuracy                   0.9995
Precision (Class 1)        0.9630
Recall (Class 1)           0.8387
F1-Score (Class 1)         0.8966
TNR                        0.9999
MCC                        0.8984
ROC-AUC                    0.9964
PR-AUC                     0.9323


###LightGBM

In [51]:
import lightgbm as lgb

In [52]:
scale_pos_weight_value = (y_train == 0).sum() / (y_train == 1).sum()

In [53]:
param_grid_lgbm = {
    'num_leaves': [31, 50, 70],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 150],
    'scale_pos_weight': [1, scale_pos_weight_value]
}

cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=101)

In [54]:
grid_search_lgbm = GridSearchCV(
    estimator=lgb.LGBMClassifier(objective='binary', random_state=101, verbosity=-1),
    param_grid=param_grid_lgbm,
    scoring='average_precision',
    cv=cv_strategy,
    n_jobs=-1
)

print("Tuning LightGBM... (Optimizing leaf-wise splits)")
grid_search_lgbm.fit(X_train, y_train)
best_lgbm_model = grid_search_lgbm.best_estimator_
print(f"Best Hyperparameters Found: {grid_search_lgbm.best_params_}")

Tuning LightGBM... (Optimizing leaf-wise splits)
Best Hyperparameters Found: {'learning_rate': 0.05, 'n_estimators': 150, 'num_leaves': 70, 'scale_pos_weight': 1}


In [55]:
predictions_lgbm = best_lgbm_model.predict(X_test)
y_scores_lgbm = best_lgbm_model.predict_proba(X_test)[:, 1]

In [56]:
tn, fp, fn, tp = confusion_matrix(y_test, predictions_lgbm).ravel()

metrics_lgbm = {
    'Accuracy': accuracy_score(y_test, predictions_lgbm),
    'Precision (Class 1)': precision_score(y_test, predictions_lgbm, pos_label=1),
    'Recall (Class 1)': recall_score(y_test, predictions_lgbm, pos_label=1),
    'F1-Score (Class 1)': f1_score(y_test, predictions_lgbm, pos_label=1),
    'TNR': tn / (tn + fp),
    'MCC': matthews_corrcoef(y_test, predictions_lgbm),
    'ROC-AUC': roc_auc_score(y_test, y_scores_lgbm),
    'PR-AUC': average_precision_score(y_test, y_scores_lgbm, pos_label=1)
}

In [57]:
lgbm_results = pd.DataFrame(pd.Series(metrics_lgbm), columns=['Best LightGBM'])
print("\n--- LightGBM Evaluation Parameters ---")
print(lgbm_results.round(4))


--- LightGBM Evaluation Parameters ---
                     Best LightGBM
Accuracy                    0.9991
Precision (Class 1)         0.8889
Recall (Class 1)            0.7742
F1-Score (Class 1)          0.8276
TNR                         0.9997
MCC                         0.8291
ROC-AUC                     0.9512
PR-AUC                      0.7862


##comparing all

In [58]:
import pandas as pd

final_comparison_df = pd.concat([
    log_regression_results,
    dt_results,
    knn_results,
    svm_results,
    rf_results,
    xgb_results,
    lgbm_results
], axis=1)

print("\n--- Final Model Comparison Matrix ---")
display(final_comparison_df.round(4))


--- Final Model Comparison Matrix ---


,Best Logistic Regression,Best Decision Tree,Best KNN,Best SVM,Best Random Forest,Best XGBoost,Best LightGBM
Accuracy,0.9821,0.9988,0.9991,0.9987,0.9990,0.9995,0.9991
Precision (Class 1),0.1293,0.7931,0.9565,0.8077,0.9545,0.9630,0.8889
Recall (Class 1),0.9677,0.7419,0.7097,0.6774,0.6774,0.8387,0.7742
F1-Score (Class 1),0.2281,0.7667,0.8148,0.7368,0.7925,0.8966,0.8276
TNR,0.9821,0.9995,0.9999,0.9996,0.9999,0.9999,0.9997
MCC,0.3503,0.7665,0.8235,0.7390,0.8037,0.8984,0.8291
ROC-AUC,0.9944,0.8709,0.9676,0.9451,0.9835,0.9964,0.9512
PR-AUC,0.7549,0.7426,0.9024,0.7133,0.9388,0.9323,0.7862
